### Step 1: Use annotated max heights to estimate how repo height varies with repo area

In [2]:
import pandas as pd
import numpy as np
import os
import pyproj

In [4]:
# make save directory
out_dir = "../out/20240905/"
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

# load in flux file first
raw_data_root = "/Users/nick/Projects/gangaSandFlux/dat/20240905/"
filename = "StockpileFlux_Points_Final_corrected.csv"
sand_flux_df = pd.read_csv(os.path.join(raw_data_root, filename))
sand_flux_df.dropna(inplace=True)
sand_flux_df.reset_index(inplace=True, drop=True)

sand_flux_df.head(5)

,Stockpile_ID,AreaName,Date,numID,Type,PointName,POINT_X,POINT_Y
0,a,Bagmara,1312018,5,base,a_Bagmara_01312018_5_base,89.414102,23.851631
1,a,Bagmara,1312018,5,base,a_Bagmara_01312018_5_base,89.414108,23.851643
2,a,Bagmara,1312018,5,base,a_Bagmara_01312018_5_base,89.414195,23.851627
3,a,Bagmara,1312018,5,base,a_Bagmara_01312018_5_base,89.414383,23.851591
4,a,Bagmara,1312018,5,base,a_Bagmara_01312018_5_base,89.414415,23.851577


In [5]:
# convert repo coordinates from lat lon to meters
projecter = pyproj.Proj(proj='utm', zone=45, ellps='WGS84', preserve_units=True)
point_tuple = projecter(sand_flux_df.loc[:, "POINT_X"], sand_flux_df.loc[:, "POINT_Y"])

# add to DF
sand_flux_df.loc[:, "point_x_m"] = point_tuple[0]-np.min(point_tuple[0])
sand_flux_df.loc[:, "point_y_m"] = point_tuple[1]-np.min(point_tuple[1])

sand_flux_df.loc[:, "AreaName"] = sand_flux_df.loc[:, "AreaName"].astype(str)
sand_flux_df.loc[:, "numID"] = sand_flux_df.loc[:, "numID"].astype(int)
sand_flux_df.loc[:, "Date"] = sand_flux_df.loc[:, "Date"].astype(str)
sand_flux_df.loc[:, "Type"] = sand_flux_df.loc[:, "Type"].astype(str)
# sand_flux_df = sand_flux_df.loc[sand_flux_df["Type"] != "nan"].reset_index(inplace=True, drop=True)


# make new ID variable
id_vec = [sand_flux_df.loc[p, "AreaName"] + "_" + sand_flux_df.loc[p, "Date"] + "_" 
                 + str(sand_flux_df.loc[p, "numID"]) for p in range(sand_flux_df.shape[0])] 
sand_flux_df.loc[:, "repoID"] = id_vec

In [7]:
def PolyArea(x,y):
    return 0.5*np.abs(np.dot(x,np.roll(y,1))-np.dot(y,np.roll(x,1)))

In [22]:
from tqdm import tqdm

# iterate through repos and calculate mh distances
repo_index = np.unique(sand_flux_df["repoID"])
mh_length_vec = []
repo_area_vec = []
point_name_vec = []
sub_ind_vec = []

for p, point_name in enumerate(tqdm(repo_index)):
    
    # filter
    df_temp = sand_flux_df.loc[sand_flux_df["repoID"]==point_name, :]
    df_temp = df_temp.reset_index(drop=True)
    
    # do we have mh info (only present if there is a plateau)?
    type_vec = df_temp["Type"].to_numpy()
    mh_flags = np.asarray([1 if "mh" in p else 0 for p in type_vec])
    base_flags = np.asarray([1 if "base" in p else 0 for p in type_vec])
    
    if np.any(mh_flags) and df_temp.shape[0] > 10 and np.any(base_flags):
        mh_index = np.unique(df_temp.loc[mh_flags==1, "Type"])
        n = len(mh_index) // 2
        
        for m, mh in enumerate(mh_index): # there can be multiple piles in the same repo
            mh_ft = np.where(type_vec==mh_index)[0]
            
            if len(mh_ft) == 2:
                # get length components
                dx = df_temp.loc[mh_ft[0], "point_x_m"] - df_temp.loc[mh_ft[1], "point_x_m"]
                dy = df_temp.loc[mh_ft[0], "point_y_m"] - df_temp.loc[mh_ft[1], "point_y_m"]
                
                # extract base points
                if n == 0:
                    key = "base"
                else:
                    key = "base" + str(m+1)
                    
                base_points_xy = df_temp.loc[type_vec==key, ["point_x_m", "point_y_m"]].to_numpy()
                assert base_points_xy.shape[0] > 0
                # calculate area
                area = PolyArea(base_points_xy[:, 0], base_points_xy[:, 1])
                
                # add info
                mh_length_vec.append(np.sqrt(dx**2 + dy**2))
                sub_ind_vec.append(m)
                point_name_vec.append(point_name)
                repo_area_vec.append(area)

  7%|▋         | 54/752 [00:00<00:01, 536.66it/s]/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_29293/3512608128.py:26: DeprecationWarning:

elementwise comparison failed; this will raise an error in the future.

/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_29293/3512608128.py:26: DeprecationWarning:

elementwise comparison failed; this will raise an error in the future.

/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_29293/3512608128.py:26: DeprecationWarning:

elementwise comparison failed; this will raise an error in the future.

/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_29293/3512608128.py:26: DeprecationWarning:

elementwise comparison failed; this will raise an error in the future.

 75%|███████▍  | 563/752 [00:01<00:00, 549.01it/s]/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_29293/3512608128.py:26: DeprecationWarning:

elementwise comparison failed; this will raise an error in the future.

/var/folders/m7/tpjtt

In [56]:
# import scipy

# repo_areas = np.asarray(repo_area_vec)
# repo_heights = np.asarray(mh_length_vec) * np.tan(np.pi/6)
# # fit height function
# # [max_h, area_hm, n]
# x0 = [10, 2e3, 1]

# def height_fun(params, repo_areas=repo_areas):
#     h = np.divide(params[0] * repo_areas**params[2], (params[1]**params[2] + repo_areas**params[2]))
#     return h

# def loss_fun(params, repo_heights=repo_heights):
#     h_hat = height_fun(params)
#     return repo_heights - h_hat

# params_fit = scipy.optimize.least_squares(loss_fun, x0)

In [69]:
import scipy

repo_areas = np.asarray(repo_area_vec)
repo_heights = np.asarray(mh_length_vec) * np.tan(np.pi/6)
# fit height function
# [max_h, area_hm, n]
x0 = [100]
max_h = np.percentile(repo_heights, 99)

def height_fun(params, max_h =max_h, repo_areas=repo_areas):
    h = np.divide(max_h * repo_areas, (params[0] + repo_areas))
    return h

def loss_fun(params, repo_heights=repo_heights):
    h_hat = height_fun(params)
    return repo_heights - h_hat

params_fit = scipy.optimize.least_squares(loss_fun, x0)

In [70]:
import plotly.graph_objects as go 

area_vec = np.linspace(0, 25000, 1000)
height_trend = height_fun(params_fit.x, repo_areas=area_vec)

fig = px.scatter(x=repo_areas, y=repo_heights)
fig.add_trace(go.Scatter(x=area_vec, y=height_trend, mode="lines"))
fig.show()

In [68]:
params_fit.x

array([729.05345885,   6.41527883])